---Sets up the target locations and authenticates to Azure Storage.

---uses the secure cloud protocol abfss:// (Azure Blob File System driver) to communicate directly with your storage container inside Azure Data Lake.


In [0]:
import fnmatch
import os
import yaml
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# --- 1. CONFIGURATION & METADATA ---
STORAGE_ACCOUNT = "metadataprojectdev"
CONTAINER_NAME = "clientalpha"
BASE_ABFSS_URL = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
TARGET_TABLE = "bronze.sale_transactions"
QUARANTINE_TABLE = "bronze.sale_transactions_quarantine"

In [0]:
# Load metadata contract
metadata_path = f"{BASE_ABFSS_URL}/data/metadata.yaml"

#cleans off any Markdown formatting wraps (````yaml), and uses the standard yaml` library to convert it into a usable Python dictionary.

metadata_text = "\n".join(row.value for row in spark.read.text(metadata_path).collect())
metadata = yaml.safe_load(metadata_text.removeprefix("```yaml\n").removesuffix("\n```"))

#2. Dynamically locates the file to process so that hardcoded file paths aren't required.
# --- SIMPLE FILE DISCOVERY & PARSING ---
source_root = f"{BASE_ABFSS_URL}/data"
available_files = [item.path for item in dbutils.fs.ls(source_root)]

# Extract first configuration block found
source = metadata.get("sources", [metadata])[0]
src_config = source["source_config"]
pattern = src_config["file_pattern"]

In [0]:
# Locate matching target file
file_path = next((f for f in available_files if fnmatch.fnmatch(os.path.basename(f), pattern)), None)
if not file_path:
    pattern_stem = os.path.splitext(pattern)[0]
    supported_exts = [".csv", ".tsv", ".jsonl", ".json", ".parquet"]
    fallback_candidates = [
        f for f in available_files
        if os.path.splitext(os.path.basename(f))[0] == pattern_stem
        and os.path.splitext(f)[1].lower() in supported_exts
    ]
    fallback_candidates.sort(key=lambda f: supported_exts.index(os.path.splitext(f)[1].lower()))
    file_path = fallback_candidates[0] if fallback_candidates else None
if not file_path:
    raise FileNotFoundError(f"No source file matched the pattern '{pattern}' in {source_root}")



In [0]:
# Parse based on file extension
file_ext = os.path.splitext(file_path)[1].lower()
if file_ext in (".csv", ".tsv"):
    raw_df = spark.read.format("csv") \
        .option("header", src_config.get("header", True)) \
        .option("delimiter", "\t" if file_ext == ".tsv" else src_config.get("delimiter", ",")) \
        .load(file_path)
elif "json" in file_ext:
    raw_df = spark.read.option("multiline", "true").json(file_path)
elif file_ext == ".parquet":
    raw_df = spark.read.parquet(file_path)
else:
    raise ValueError(f"Unsupported extension: {file_ext}")
# Loops through each column defined in the metadata schema file to cast and map structural types automatically.
# --- 3. MAPPING & CASTING ---
select_columns = []
for col_meta in source["schema_definition"]["columns"]:
    src_col = col_meta["name"]
    tgt_col = col_meta.get("target_column", src_col)
    d_type = col_meta["data_type"]
    
    if "DECIMAL" in d_type:
        expr = F.col(src_col).cast(d_type)
    elif d_type == "DATE":
        expr = F.to_date(F.col(src_col), col_meta.get("format", "yyyy-MM-dd"))
    else:
        expr = F.col(src_col).cast("string")
    select_columns.append(expr.alias(tgt_col))

mapped_df = raw_df.select(*select_columns)



In [0]:
#Instantiates auditing fields to track data cleanliness.
# --- 4. QUALITY KICKOUT LOGIC (QUARANTINE) ---
name_map = {c["name"]: c.get("target_column", c["name"]) for c in source["schema_definition"]["columns"]}
validated_df = mapped_df.withColumn("is_quarantined", F.lit(False)) \
    .withColumn("quarantine_reason", F.lit(None).cast("string"))

#Marks any transaction row where the numeric value drops below or matches the minimum allowed limit.
for rule in source.get("quality_rules", []):
    tgt_col = name_map.get(rule["column"], rule["column"])
    
    if rule["check_type"] == "GREATER_THAN":
        fail_condition = F.col(tgt_col) <= rule["threshold"]
    elif rule["check_type"] == "UNIQUE":
        fail_condition = F.count("*").over(Window.partitionBy(tgt_col)) > 1
    else:
        continue

    reason = f"{rule['rule_id']}: Fail on {tgt_col}"
    validated_df = validated_df.withColumn(
        "quarantine_reason",
        F.when(fail_condition, F.concat_ws(" | ", F.col("quarantine_reason"), F.lit(reason))).otherwise(F.col("quarantine_reason"))
    ).withColumn("is_quarantined", F.when(fail_condition, F.lit(True)).otherwise(F.col("is_quarantined")))



In [0]:
# Split datasets
valid_df = validated_df.filter("is_quarantined = false").drop("is_quarantined", "quarantine_reason")
quarantine_df = validated_df.filter("is_quarantined = true")


# Write valid data
load_mode = "overwrite" if metadata.get("delivery_metadata", {}).get("load_type") == "OVERWRITE" else "append"
if load_mode == "append":
    try:
        spark.sql(f"DESCRIBE DETAIL {TARGET_TABLE}")
    except Exception as e:
        if "DELTA_TABLE_NOT_FOUND" in str(e) or "DELTA_PATH_DOES_NOT_EXIST" in str(e):
            load_mode = "overwrite"
        else:
            raise
valid_df.write.format("delta").mode(load_mode).saveAsTable(TARGET_TABLE)
display(valid_df)



In [0]:
# Write quarantine data if errors exist
if quarantine_df.limit(1).count() > 0:
    quarantine_mode = "append"
    try:
        spark.sql(f"DESCRIBE DETAIL {QUARANTINE_TABLE}")
    except Exception as e:
        if (
            "DELTA_TABLE_NOT_FOUND" in str(e)
            or "DELTA_PATH_DOES_NOT_EXIST" in str(e)
            or "TABLE_OR_VIEW_NOT_FOUND" in str(e)
        ):
            quarantine_mode = "overwrite"
        else:
            raise
    quarantine_df.write.format("delta").mode(quarantine_mode).saveAsTable(QUARANTINE_TABLE)

print(f"[SUCCESS] Pipelines updated. Target: {TARGET_TABLE}")

